# EDA Pipeline Runner (Kaggle)

Этот ноутбук запускает пайплайн EDA для задачи "Не слышу тебя" в среде Kaggle.

## 0. Подготовка исходников
Добавьте к ноутбуку два датасета:
1. `vseros-a-repo` (или аналогичный) с папкой `Vseros_A`.
2. Датасет с аудио (`train_opus/`, `test_opus/`) и `word_bounds.json`.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

input_root = Path('/kaggle/input')
working_root = Path('/kaggle/working')
repo_dst = working_root / 'Vseros_A'

repo_src = None
for candidate in input_root.glob('*'):
    test_path = candidate / 'Vseros_A'
    if test_path.exists():
        repo_src = test_path
        break

if repo_src is None:
    raise FileNotFoundError('Добавьте датасет с репозиторием Vseros_A в /kaggle/input')

if repo_dst.exists():
    shutil.rmtree(repo_dst)
shutil.copytree(repo_src, repo_dst)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_dst / 'EDA/requirements.txt')])

print('Репозиторий скопирован в', repo_dst)


## 1. Переход в рабочий каталог


In [ ]:
import os
os.chdir('/kaggle/working/Vseros_A')
print('Текущий каталог:', os.getcwd())


## 2. Копирование аудио из /kaggle/input и настройка конфигурации


In [ ]:
from pathlib import Path
import shutil
import yaml

input_root = Path('/kaggle/input')
dataset_src = None
for candidate in input_root.glob('*'):
    if (candidate / 'train_opus').exists() and (candidate / 'test_opus').exists():
        dataset_src = candidate
        break

if dataset_src is None:
    raise FileNotFoundError('Не найден датасет с train_opus/test_opus в /kaggle/input')

workspace = Path('/kaggle/working/eda_workspace')
raw_train_dst = workspace / 'data/raw/train_opus'
raw_test_dst = workspace / 'data/raw/test_opus'
meta_dst = workspace / 'data/meta'

shutil.rmtree(workspace, ignore_errors=True)
raw_train_dst.mkdir(parents=True, exist_ok=True)
raw_test_dst.mkdir(parents=True, exist_ok=True)
meta_dst.mkdir(parents=True, exist_ok=True)

def copy_tree(src: Path, dst: Path):
    for item in src.rglob('*'):
        rel = item.relative_to(src)
        target = dst / rel
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)

copy_tree(dataset_src / 'train_opus', raw_train_dst)
copy_tree(dataset_src / 'test_opus', raw_test_dst)

wb_src = dataset_src / 'word_bounds.json'
if not wb_src.exists():
    raise FileNotFoundError('Ожидался word_bounds.json в корне датасета данных')
shutil.copy2(wb_src, meta_dst / 'word_bounds.json')

default_cfg_path = Path('EDA/configs/default.yaml')
cfg = yaml.safe_load(default_cfg_path.read_text())

cfg['paths'] = {
    'project_root': '/kaggle/working',
    'data_root': '/kaggle/working/eda_workspace/data',
    'raw_train': '/kaggle/working/eda_workspace/data/raw/train_opus',
    'raw_test': '/kaggle/working/eda_workspace/data/raw/test_opus',
    'meta': '/kaggle/working/eda_workspace/data/meta',
    'interim_wav': '/kaggle/working/eda_workspace/data/interim/wav16k_mono',
    'tables': '/kaggle/working/eda_workspace/data/tables',
    'samples': '/kaggle/working/eda_workspace/data/samples',
    'reports': '/kaggle/working/eda_workspace/reports',
    'figs': '/kaggle/working/eda_workspace/reports/figs',
    'env_log': '/kaggle/working/eda_workspace/data/meta/env_versions.txt',
    'readme_short': '/kaggle/working/eda_workspace/data/meta/README_EDA.md'
}
cfg['downloads'] = {}
cfg['dataset']['word_bounds_path'] = '/kaggle/working/eda_workspace/data/meta/word_bounds.json'
cfg['report']['summary_path'] = '/kaggle/working/eda_workspace/reports/EDA_Summary.md'
cfg['report']['summary_pdf_path'] = '/kaggle/working/eda_workspace/reports/EDA_Summary.pdf'

kaggle_cfg_path = Path('EDA/configs/kaggle.yaml')
kaggle_cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

print('Данные скопированы из', dataset_src)
print('Создана конфигурация', kaggle_cfg_path)


## 3. Запуск пайплайна


In [ ]:
%%bash
set -e
cd /kaggle/working/Vseros_A
python -m EDA.run prepare --config EDA/configs/kaggle.yaml --force
python -m EDA.run download --config EDA/configs/kaggle.yaml --force
python -m EDA.run inventory --config EDA/configs/kaggle.yaml --force
python -m EDA.run labels --config EDA/configs/kaggle.yaml --force
python -m EDA.run convert --config EDA/configs/kaggle.yaml --force
python -m EDA.run vad --config EDA/configs/kaggle.yaml --force
python -m EDA.run negatives --config EDA/configs/kaggle.yaml --force
python -m EDA.run duplicates --config EDA/configs/kaggle.yaml --force
python -m EDA.run cv --config EDA/configs/kaggle.yaml --force
python -m EDA.run windowing --config EDA/configs/kaggle.yaml --force
python -m EDA.run slices --config EDA/configs/kaggle.yaml --force
python -m EDA.run report --config EDA/configs/kaggle.yaml --force


## 4. Артефакты


In [ ]:
from pathlib import Path
root = Path('/kaggle/working/eda_workspace')
for path in sorted(root.glob('**/*')):
    if path.is_file():
        print(path.relative_to('/kaggle/working'))
